In [5]:
from pynq import Overlay, allocate
import numpy as np, time

In [7]:
# ---- EDIT THESE PATHS IF NEEDED ----
BIT_PATH = "/home/xilinx/jupyter_notebooks/conv.bit"
HWH_PATH = BIT_PATH.replace(".bit", ".hwh")   # HWH auto-loaded from same path/name
DATA_DIR = "/home/xilinx/jupyter_notebooks/"   # folder with the 4 .bin files

# ---- Layer sizes (must match conv.hpp) ----
M, I, K, F, O, S = 3, 228, 11, 64, 55, 4

ol = Overlay(BIT_PATH)
ol.download()
print("Loaded overlay. IPs found:", list(ol.ip_dict.keys()))

ip_name = None
for k in ol.ip_dict.keys():
    if "conv" in k.lower():
        ip_name = k
        break
if ip_name is None:
    for k in ol.ip_dict.keys():
        if "processing_system7" not in k:
            ip_name = k
            break

ip = getattr(ol, ip_name)
print("Using IP:", ip_name)
print(ip.register_map)

# Helper: write 64-bit pointer split into _1 (low 32) and _2 (high 32)
def write_ptr64(ip, base_name, addr):
    low  = int(addr & 0xFFFFFFFF)
    high = int((addr >> 32) & 0xFFFFFFFF)
    setattr(ip.register_map, base_name + "_1", low)
    setattr(ip.register_map, base_name + "_2", high)

rm = ip.register_map
ptr_fields = [f for f in dir(rm) if f.endswith("_1") or f.endswith("_2")]
print("Pointer fields present:", ptr_fields)

Loaded overlay. IPs found: ['conv_0', 'processing_system7_0']
Using IP: conv_0
RegisterMap {
  CTRL = Register(AP_START=0, AP_DONE=0, AP_IDLE=1, AP_READY=0, RESERVED_1=0, AUTO_RESTART=0, RESERVED_2=0, INTERRUPT=0, RESERVED_3=0),
  GIER = Register(Enable=0, RESERVED=0),
  IP_IER = Register(CHAN0_INT_EN=0, CHAN1_INT_EN=0, RESERVED_0=0),
  IP_ISR = Register(CHAN0_INT_ST=0, CHAN1_INT_ST=0, RESERVED_0=0),
  input_fm_1 = Register(input_fm=write-only),
  input_fm_2 = Register(input_fm=write-only),
  weights_1 = Register(weights=write-only),
  weights_2 = Register(weights=write-only),
  biases_1 = Register(biases=write-only),
  biases_2 = Register(biases=write-only),
  output_fm_1 = Register(output_fm=write-only),
  output_fm_2 = Register(output_fm=write-only)
}
Pointer fields present: ['biases_1', 'biases_2', 'input_fm_1', 'input_fm_2', 'output_fm_1', 'output_fm_2', 'weights_1', 'weights_2']


In [8]:
x = np.fromfile(DATA_DIR + "conv_input.bin", dtype=np.float32)
n = int(round((x.size / M) ** 0.5))          # 226 or 228
x = x.reshape(M, n, n)
if n < I:                                     # pad up to 228x228 if the file is 226x226
    p = (I - n) // 2
    x = np.pad(x, ((0, 0), (p, p), (p, p)))
assert x.shape == (M, I, I), x.shape

w   = np.fromfile(DATA_DIR + "conv_weights.bin", dtype=np.float32).reshape(F, M, K, K)
b   = np.fromfile(DATA_DIR + "conv_biases.bin",  dtype=np.float32)
ref = np.fromfile(DATA_DIR + "conv_output.bin",  dtype=np.float32).reshape(F, O, O)

in_buf  = allocate((M, I, I),    dtype=np.float32)
w_buf   = allocate((F, M, K, K), dtype=np.float32)
b_buf   = allocate((F,),         dtype=np.float32)
out_buf = allocate((F, O, O),    dtype=np.float32)

in_buf[:] = x
w_buf[:]  = w
b_buf[:]  = b
out_buf.fill(0)
for buf in (in_buf, w_buf, b_buf, out_buf):
    buf.flush()

print("input", in_buf.shape, "weights", w_buf.shape, "biases", b_buf.shape, "output", out_buf.shape)


input (3, 228, 228) weights (64, 3, 11, 11) biases (64,) output (64, 55, 55)


In [9]:
# ================================================
# Cell 3: CPU (PS) baseline - NumPy, stride 4, bias + ReLU
# ================================================
from numpy.lib.stride_tricks import sliding_window_view

def conv_cpu(x_in, w_in, b_in):
    win = sliding_window_view(x_in, (K, K), axis=(1, 2))[:, ::S, ::S]     # (M,O,O,K,K)
    out = np.tensordot(w_in, win, axes=([1, 2, 3], [0, 3, 4])) + b_in[:, None, None]
    return np.maximum(out, 0).astype(np.float32)

def time_fn_ms(fn, runs=3, warmup=1):
    for _ in range(warmup):
        fn()
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        fn()
        t1 = time.perf_counter()
        times.append((t1 - t0) * 1e3)
    return float(np.median(times)), times

cpu_ms, cpu_runs = time_fn_ms(lambda: conv_cpu(x, w, b), runs=5, warmup=1)
print(f"CPU NumPy median over 5: {cpu_ms:.2f} ms")


CPU NumPy median over 5: 264.21 ms


In [15]:
write_ptr64(ip, "input_fm",  in_buf.physical_address)
write_ptr64(ip, "weights",   w_buf.physical_address)
write_ptr64(ip, "biases",    b_buf.physical_address)
write_ptr64(ip, "output_fm", out_buf.physical_address)

print("CTRL before:", hex(ip.read(0x00)))
while (ip.read(0x00) & 0x4) == 0:        # wait until AP_IDLE
    pass
ip.read(0x00)                             # clear any stale done flag

out_buf.fill(0)
out_buf.flush()
t0 = time.perf_counter()
ip.write(0x00, 1)                         # ap_start
while (ip.read(0x00) & 0x2) == 0:         # wait AP_DONE
    pass
t1 = time.perf_counter()
out_buf.invalidate()

fpga_ms = (t1 - t0) * 1e3
print(f"FPGA time (kernel only): {fpga_ms:.3f} ms  ({fpga_ms/1e3:.2f} s)")
print("CTRL after:", hex(ip.read(0x00)))
print("output nonzero count:", int(np.count_nonzero(out_buf)))

CTRL before: 0x1
FPGA time (kernel only): 98777.840 ms  (98.78 s)
CTRL after: 0x4
output nonzero count: 64765


In [16]:
# ================================================
# Cell 5: Correctness & speedup
# ================================================
out_fpga = np.array(out_buf).reshape(F, O, O)

mse_ref = float(np.mean((out_fpga - ref) ** 2))
mse_cpu = float(np.mean((out_fpga - conv_cpu(x, w, b)) ** 2))
max_abs = float(np.max(np.abs(out_fpga - ref)))
print(f"MSE (FPGA vs reference .bin): {mse_ref:.3e}")
print(f"MSE (FPGA vs CPU NumPy)     : {mse_cpu:.3e}")
print(f"Max |FPGA - ref|            : {max_abs:.3e}")
print("PASS" if mse_ref < np.exp(-4) else "FAIL")

ops = 2.0 * F * O * O * M * K * K       # multiply + add
cpu_gflops  = (ops / (cpu_ms / 1e3)) / 1e9
fpga_gflops = (ops / (fpga_ms / 1e3)) / 1e9
speedup = cpu_ms / fpga_ms if fpga_ms > 0 else float("inf")

HLS_EST_S = 61.735                      # from the Vitis HLS report (6173528582 cycles @ 100 MHz)
print("\n=== Performance ===")
print(f"CPU time            : {cpu_ms:.3f} ms")
print(f"FPGA time           : {fpga_ms:.3f} ms")
print(f"Vitis HLS estimate  : {HLS_EST_S*1e3:.0f} ms  (measured/estimate = {fpga_ms/(HLS_EST_S*1e3):.2f}x)")
print(f"CPU perf : {cpu_gflops:.3f} GFLOP/s")
print(f"FPGA perf: {fpga_gflops:.3f} GFLOP/s")
print(f"Speedup (CPU/FPGA) = {speedup:.4f}x")


MSE (FPGA vs reference .bin): 2.484e-13
MSE (FPGA vs CPU NumPy)     : 1.216e-13
Max |FPGA - ref|            : 1.335e-05
PASS

=== Performance ===
CPU time            : 264.212 ms
FPGA time           : 98777.840 ms
Vitis HLS estimate  : 61735 ms  (measured/estimate = 1.60x)
CPU perf : 0.532 GFLOP/s
FPGA perf: 0.001 GFLOP/s
Speedup (CPU/FPGA) = 0.0027x


In [19]:
# ================================================
# Cell 6: Peek at results
# ================================================
def peek(name, X, c=5):
    print(f"\n{name} (channel 0, top-left {c}x{c})")
    print(X[0, :c, :c])

peek("FPGA output", out_fpga)
peek("Reference",   ref)

for buf in (in_buf, w_buf, b_buf, out_buf):
    buf.freebuffer()



FPGA output (channel 0, top-left 5x5)
[[0.7140566  0.         0.         0.         0.        ]
 [0.04751142 0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.        ]]

Reference (channel 0, top-left 5x5)
[[0.71405655 0.         0.         0.         0.        ]
 [0.0475114  0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.        ]]


In [20]:
C_cpu = conv_cpu(x, w, b)

def peek_tensor(name, X, c=8):
    print(f"\n{name} shape={X.shape}")
    if X.ndim == 3:
        print(X[0, :c, :c])
    elif X.ndim == 2:
        print(X[:c, :c])
    else:
        print(X[:c])

peek_tensor("input_fm (padded 228x228)", x)
peek_tensor("weights[0]", w[0])
peek_tensor("output_fpga", out_fpga)
peek_tensor("reference (.bin)", ref)
peek_tensor("cpu_output", C_cpu)

print("\nnp.allclose(FPGA, ref):", np.allclose(out_fpga, ref, atol=1e-4))
print("np.allclose(FPGA, CPU):", np.allclose(out_fpga, C_cpu, atol=1e-4))

for buf in (in_buf, w_buf, b_buf, out_buf):
    buf.freebuffer()


input_fm (padded 228x228) shape=(3, 228, 228)
[[ 0.          0.          0.          0.          0.          0.          0.          0.        ]
 [ 0.          0.          0.          0.          0.          0.          0.          0.        ]
 [ 0.          0.         -0.8335474  -0.88492167 -0.8677969  -0.67942464 -0.59380084 -0.5424266 ]
 [ 0.          0.         -0.79929787 -0.79929787 -0.78217316 -0.69654936 -0.55955136 -0.5253019 ]
 [ 0.          0.         -0.8335474  -0.8335474  -0.79929787 -0.69654936 -0.59380084 -0.4739276 ]
 [ 0.          0.         -0.8335474  -0.81642264 -0.7650484  -0.67942464 -0.57667613 -0.5081771 ]
 [ 0.          0.         -0.7650484  -0.7650484  -0.6622999  -0.5424266  -0.5081771  -0.4739276 ]
 [ 0.          0.         -0.6622999  -0.6622999  -0.5253019  -0.43967807 -0.43967807 -0.45680285]]

weights[0] shape=(3, 11, 11)
[[ 0.11863963  0.09406868  0.09543519  0.10515065  0.10291012  0.0672244   0.05046584  0.05012476]
 [ 0.07488242  0.03894044  0.05